Language: **English (default)** | [Polski](lab.ipynb)

# Lab 04 -- an experimental comparison of optimization strategies

Companion material to [`README.md`](README.md). Regression on data with
deliberately different feature scales -- you compare SGD, momentum, and
Adam, study the effect of the learning rate, diagnose a failed training
run, and evaluate honestly on the test set.

In [ ]:
import matplotlib.pyplot as plt  # noqa: F401 - used in sections 3/4/6
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Data (given)

Three features with different scales: $x_1\in[-1,1]$, $x_2\in[-40,40]$,
$x_3\in[-3,3]$; $y = 3\sin(1.5 x_1) + 0.05 x_2 + 0.8 x_3 + \varepsilon$.
A 70/15/15 split (train/validation/test).

In [ ]:
def make_data(n, seed):
    g = torch.Generator().manual_seed(seed)
    x1 = torch.empty(n).uniform_(-1, 1, generator=g)
    x2 = torch.empty(n).uniform_(-40, 40, generator=g)
    x3 = torch.empty(n).uniform_(-3, 3, generator=g)
    noise = torch.randn(n, generator=g) * 0.5
    y = 3.0 * torch.sin(1.5 * x1) + 0.05 * x2 + 0.8 * x3 + noise
    X = torch.stack([x1, x2, x3], dim=1)
    return X, y.unsqueeze(-1)


X, y = make_data(1600, seed=0)
n = X.shape[0]
perm = torch.randperm(n, generator=torch.Generator().manual_seed(1))
n_val = int(0.15 * n)
n_test = int(0.15 * n)
val_idx = perm[:n_val]
test_idx = perm[n_val : n_val + n_test]
train_idx = perm[n_val + n_test :]

X_train, y_train = X[train_idx].to(device), y[train_idx].to(device)
X_val, y_val = X[val_idx].to(device), y[val_idx].to(device)
X_test, y_test = X[test_idx].to(device), y[test_idx].to(device)

print(f"N_train={len(train_idx)}, N_val={len(val_idx)}, N_test={len(test_idx)}")
print(f"feature std deviations: {X_train.std(dim=0).tolist()}")

## 2. Training loop -- your task

In [ ]:
def make_model(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(3, 32), nn.ReLU(), nn.Linear(32, 1)).to(device)


def iterate_minibatches(X, y, batch_size, seed):
    """Explicit mini-batch construction: permute indices, slice into chunks."""
    g = torch.Generator().manual_seed(seed)
    n = X.shape[0]
    perm = torch.randperm(n, generator=g)
    for i in range(0, n, batch_size):
        idx = perm[i : i + batch_size]
        yield X[idx], y[idx]

In [ ]:
def train_model(opt_name, lr, epochs, batch_size=32, model_seed=42, data_seed=123, momentum=0.9):
    """Trains a model (opt_name: 'sgd'|'momentum'|'adam'), returns
    (model, train_losses, val_losses) -- MSE loss after every epoch.
    """
    # TODO:
    # 1. model = make_model(model_seed)
    # 2. build the optimizer: torch.optim.SGD(..., lr=lr) or
    #    torch.optim.SGD(..., lr=lr, momentum=momentum) or torch.optim.Adam(..., lr=lr)
    #    depending on opt_name
    # 3. loss_fn = nn.MSELoss()
    # 4. in a loop over epochs: for each mini-batch from iterate_minibatches
    #    (use seed=data_seed+epoch, so the batch order changes between
    #    epochs but stays reproducible) perform:
    #    zero_grad() -> forward -> loss -> backward() -> step()
    # 5. after every epoch compute and record the loss on X_train/y_train and
    #    X_val/y_val (under torch.no_grad())
    raise NotImplementedError("TODO: train_model")

In [ ]:
baseline_val = ((y_val - y_train.mean()) ** 2).mean().item()
baseline_test = ((y_test - y_train.mean()) ** 2).mean().item()
print(f"Baseline (training mean): val MSE={baseline_val:.4f}  test MSE={baseline_test:.4f}")

## 3. Comparison at a shared learning rate -- your task

In [ ]:
# TODO: call train_model for "sgd", "momentum", "adam" with lr=0.01, epochs=20.
# Collect the results (e.g. in a dict common_results[name] = (tl, vl)) and plot
# the training and validation loss curves for all three (2 plots side by side,
# or one with 3 lines each).
raise NotImplementedError("TODO: comparison at a shared lr")

## 4. Learning-rate sweep -- your task

In [ ]:
# TODO: for SGD test lr from [0.0001, 0.001, 0.01] (epochs=20 each),
# print the final validation loss for each. Do the same for momentum
# (momentum=0.9) with lr from [0.00005, 0.0005, 0.005].
raise NotImplementedError("TODO: learning-rate sweep")

## 5. Diagnosis -- your task

**Task (Markdown, 4-6 sentences):** in section 3, at least one optimizer
(other than Adam) at `lr=0.01` probably learned nothing sensible (the
loss may have exploded to `nan`). Based on the results from section 4 and
what you know from the lecture about different feature scales and the
loss's curvature: diagnose **why** exactly `lr=0.01` failed for that
optimizer, but not for Adam. "lr was too large" is not enough -- explain
what it is about *this specific data* that makes SGD/momentum so
sensitive to the choice of `lr`.

_(Your answer here.)_

## 6. Final evaluation -- your task

Choose **one, best by validation loss** `lr` for each of the three
optimizers (based on section 4 -- for Adam you may use `lr=0.01` from
section 3). Train each with the chosen `lr` (`epochs=25`), and then
evaluate **once** on the test set.

In [ ]:
# TODO:
# 1. define best_lrs = {"sgd": ..., "momentum": ..., "adam": 0.01} based on
#    your results from section 4
# 2. for each optimizer: train_model(..., lr=best_lrs[name], epochs=25)
# 3. compute the loss on X_test/y_test (under torch.no_grad(), nn.MSELoss())
# 4. print a table: optimizer, lr, training loss, validation loss, TEST loss
raise NotImplementedError("TODO: final evaluation")

## 7. Conclusions -- your task

Answer (Markdown, justified with real numbers from section 6):

1. Which optimizer achieved the lowest test loss? Does this result match
   what the lecture taught you about problems ill-conditioned by
   different feature scales?
2. If you compared the same three optimizers on a problem **without** a
   difference in feature scales (all features normalized) -- would you
   expect a similarly large difference between them? What is your
   prediction based on?

_(Your answer here.)_